# Clifford+T inference

In [1]:
import json
from pathlib import Path

import numpy as np
import torch

from policy import Conv1dPolicy
from qiskit_gym.envs.clifford_t import CliffordTEnv

from utils import restore_tuples, CliffordTSolver

### Load config and model

In [2]:
num_qubits = 3

config = json.load(open(f"cliffordt_{num_qubits}q.json"), object_hook=restore_tuples)
config["env"]["depth_slope"] = 2
config["env"]["max_steps"] = 256

In [3]:
env = CliffordTEnv(**config["env"])
policy = Conv1dPolicy(env.observation_space.shape, env.action_space.n, **config["policy"], device="cpu").eval()
policy.load_state_dict(torch.load(f"cliffordt_{num_qubits}q.pt", map_location="cpu", weights_only=True))
print(f"Loaded {num_qubits}q model; obs={env.observation_space.shape}, actions={env.action_space.n}")

Loaded 3q model; obs=(48, 8192), actions=126


In [4]:
solver = CliffordTSolver(env, policy, device="cpu")

### Run model

In [5]:
import pandas as pd
df = pd.read_parquet(f"benchmarks/benchmark_{num_qubits}q.parquet", engine="pyarrow", dtype_backend="pyarrow")

In [6]:
input_tcount = 30
df[df['input_tcount']==input_tcount].head(3)

,u_id,algorithm,success,correct,input_tcount,output_tcount,time,improvement,input_sequence,output_sequence
10098,2900,Greedy,False,False,30,0,148.559657,1.0,[27 54 5 30 35 63 40 26 14 58 17 43 52 41 6 ...,[]
10099,2900,Sample_10,True,True,30,32,1150.707968,-0.066667,[27 54 5 30 35 63 40 26 14 58 17 43 52 41 6 ...,[ 27 54 5 107 35 40 30 63 26 14 17 ...
10100,2900,Sample_100,True,True,30,28,11361.500043,0.066667,[27 54 5 30 35 63 40 26 14 58 17 43 52 41 6 ...,[ 27 54 107 5 35 40 63 30 26 14 17 ...


In [7]:
input_seq = df[df['input_tcount']==input_tcount].head(1)['input_sequence'].values[0]
print(input_seq)

[27, 54, 5, 30, 35, 63, 40, 26, 14, 58, 17, 43, 52, 41, 6, 35, 17, 39, 3, 62, 4, 32, 39, 12, 12, 30, 22, 39, 19, 43]


In [8]:
actions, success, reward = solver.solve(input_seq, n=100, temperature=1.0)
print(f"success={success}, reward={reward}, output_tcount={len(actions)}")

success=True, reward=0.947265625, output_tcount=28


In [9]:
print(solver.actions_to_channels(actions))

[107, 27, 54, 5, 35, 40, 83, 63, 30, 26, 86, 103, 14, 94, 58, 107, 17, 43, 41, 6, 52, 35, 17, 39, 3, 72, 96, 62]
